## Ejercicio 1

In [13]:
# cliente HDFS
from hdfs import InsecureClient
import json

client = InsecureClient(
    "http://10.22.87.215:50070",
    user="iabd-user15"
)

In [14]:
# 1.1 crear pedidos.json con 20 pedidos
pedidos = [
    {"id_pedido": 1, "cliente": "Ane", "producto": "Portátil", "categoria": "Informática", "cantidad": 1, "precio_unitario": 850, "fecha": "2026-10-19"},
    {"id_pedido": 2, "cliente": "Jon", "producto": "Ratón", "categoria": "Accesorios", "cantidad": 2, "precio_unitario": 25, "fecha": "2026-10-19"},
    {"id_pedido": 3, "cliente": "Mikel", "producto": "Teclado mecánico", "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 79.99, "fecha": "2026-10-20"},
    {"id_pedido": 4, "cliente": "Leire", "producto": "Monitor 27 pulgadas", "categoria": "Informática", "cantidad": 2, "precio_unitario": 219.5, "fecha": "2026-10-20"},
    {"id_pedido": 5, "cliente": "Jon", "producto": "Alfombrilla gaming", "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 18.9, "fecha": "2026-10-21"},
    {"id_pedido": 6, "cliente": "Nerea", "producto": "Auriculares Bluetooth", "categoria": "Audio", "cantidad": 1, "precio_unitario": 64.95, "fecha": "2026-10-21"},
    {"id_pedido": 7, "cliente": "Ane", "producto": "Disco SSD 1 TB", "categoria": "Almacenamiento", "cantidad": 2, "precio_unitario": 89.99, "fecha": "2026-10-22"},
    {"id_pedido": 8, "cliente": "Iker", "producto": "Webcam Full HD", "categoria": "Periféricos", "cantidad": 1, "precio_unitario": 42.5, "fecha": "2026-10-22"},
    {"id_pedido": 9, "cliente": "Maialen", "producto": "Impresora multifunción", "categoria": "Oficina", "cantidad": 1, "precio_unitario": 129.99, "fecha": "2026-10-23"},
    {"id_pedido": 10, "cliente": "Unai", "producto": "Cable USB-C", "categoria": "Accesorios", "cantidad": 3, "precio_unitario": 9.95, "fecha": "2026-10-23"},
    {"id_pedido": 11, "cliente": "Ane", "producto": "Mochila para portátil", "categoria": "Complementos", "cantidad": 1, "precio_unitario": 39.9, "fecha": "2026-10-24"},
    {"id_pedido": 12, "cliente": "Jon", "producto": "Monitor 24 pulgadas", "categoria": "Informática", "cantidad": 1, "precio_unitario": 149.99, "fecha": "2026-10-24"},
    {"id_pedido": 13, "cliente": "Nerea", "producto": "Micrófono USB", "categoria": "Audio", "cantidad": 1, "precio_unitario": 54.9, "fecha": "2026-10-25"},
    {"id_pedido": 14, "cliente": "Mikel", "producto": "Tarjeta gráfica", "categoria": "Componentes", "cantidad": 1, "precio_unitario": 479.99, "fecha": "2026-10-25"},
    {"id_pedido": 15, "cliente": "Leire", "producto": "Hub USB-C", "categoria": "Accesorios", "cantidad": 2, "precio_unitario": 32.95, "fecha": "2026-10-26"},
    {"id_pedido": 16, "cliente": "Iker", "producto": "Disco duro externo 2 TB", "categoria": "Almacenamiento", "cantidad": 1, "precio_unitario": 74.99, "fecha": "2026-10-26"},
    {"id_pedido": 17, "cliente": "Maialen", "producto": "Silla ergonómica", "categoria": "Mobiliario", "cantidad": 1, "precio_unitario": 189.0, "fecha": "2026-10-27"},
    {"id_pedido": 18, "cliente": "Unai", "producto": "Teclado inalámbrico", "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 34.99, "fecha": "2026-10-28"},
    {"id_pedido": 19, "cliente": "Mikel", "producto": "Memoria RAM 16 GB", "categoria": "Componentes", "cantidad": 2, "precio_unitario": 45.0, "fecha": "2026-10-29"},
    {"id_pedido": 20, "cliente": "Nerea", "producto": "Tablet 10 pulgadas", "categoria": "Dispositivos móviles", "cantidad": 1, "precio_unitario": 299.99, "fecha": "2026-10-30"},
]

# 1.2 subir pedidos.json a HDFS
with client.write(
    '/user/iabd-user15/pedidos/entrada/pedidos.json',
    encoding='utf-8',
    overwrite=True
) as writer:
    writer.write(json.dumps(pedidos, ensure_ascii=False, indent=4))

print(len(pedidos), "pedidos subidos")

20 pedidos subidos


In [15]:
# 2.2 leer pedidos.json desde HDFS
with client.read(
    '/user/iabd-user15/pedidos/entrada/pedidos.json',
    encoding='utf-8'
) as reader:
    pedidos = json.load(reader)

print(len(pedidos), "pedidos leidos")

20 pedidos leidos


## Ejercicio 2

In [16]:
# 2.1 conectarse al broker Kafka
from confluent_kafka import Producer
import socket

conf = {'bootstrap.servers':  "10.22.87.215:9092",
        'client.id': socket.gethostname()}

producer = Producer(conf)
topic = "pedidos_iabd-user15"

In [17]:
# 2.5 mostrar la confirmacion de entrega o el error
def confirmar(err, msg):
    if err:
        print("Error al enviar:", err)
    else:
        print("Enviado:", msg.value().decode("utf-8"))

# 2.3 enviar cada pedido como mensaje JSON | 2.4 clave = id_pedido
for pedido in pedidos:
    producer.produce(
        topic,
        key=str(pedido["id_pedido"]),
        value=json.dumps(pedido, ensure_ascii=False),
        callback=confirmar
    )

producer.flush(15)

Enviado: {"id_pedido": 1, "cliente": "Ane", "producto": "Portátil", "categoria": "Informática", "cantidad": 1, "precio_unitario": 850, "fecha": "2026-10-19"}
Enviado: {"id_pedido": 2, "cliente": "Jon", "producto": "Ratón", "categoria": "Accesorios", "cantidad": 2, "precio_unitario": 25, "fecha": "2026-10-19"}
Enviado: {"id_pedido": 3, "cliente": "Mikel", "producto": "Teclado mecánico", "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 79.99, "fecha": "2026-10-20"}
Enviado: {"id_pedido": 4, "cliente": "Leire", "producto": "Monitor 27 pulgadas", "categoria": "Informática", "cantidad": 2, "precio_unitario": 219.5, "fecha": "2026-10-20"}
Enviado: {"id_pedido": 5, "cliente": "Jon", "producto": "Alfombrilla gaming", "categoria": "Accesorios", "cantidad": 1, "precio_unitario": 18.9, "fecha": "2026-10-21"}
Enviado: {"id_pedido": 6, "cliente": "Nerea", "producto": "Auriculares Bluetooth", "categoria": "Audio", "cantidad": 1, "precio_unitario": 64.95, "fecha": "2026-10-21"}
Enviado: {

0

## Ejercicio 3

In [18]:
# 3.1 grupo de consumidores propio y suscripcion al topico
from confluent_kafka import Consumer

conf = {
    'bootstrap.servers': '10.22.87.215:9092',
    'group.id': 'grupo_pedidos_iabd-user15',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False
}

consumer = Consumer(conf)
consumer.subscribe(['pedidos_iabd-user15'])

pedidos_recibidos = {}

In [19]:
# 3.4 controlar errores y establecer un tiempo maximo de espera
try:
    while True:
        msg = consumer.poll(timeout=5.0)

        if msg is None:
            print("No hay mensajes")
            break

        if msg.error():
            print("Error:", msg.error())
            continue

        # 3.2 convertir cada JSON en un diccionario
        pedido = json.loads(msg.value().decode("utf-8"))
        pedidos_recibidos[pedido["id_pedido"]] = pedido

finally:
    consumer.close()

# 3.3 mostrar los pedidos recibidos
for pedido in pedidos_recibidos.values():
    print(pedido["id_pedido"], pedido["cliente"], pedido["producto"])

# 3.4 comprobar que se han recuperado los 20
if len(pedidos_recibidos) == 20:
    print("OK: 20 pedidos recuperados")
else:
    print("FALLO:", len(pedidos_recibidos), "pedidos recuperados")

No hay mensajes
1 Ane Portátil
2 Jon Ratón
3 Mikel Teclado mecánico
4 Leire Monitor 27 pulgadas
5 Jon Alfombrilla gaming
6 Nerea Auriculares Bluetooth
7 Ane Disco SSD 1 TB
8 Iker Webcam Full HD
9 Maialen Impresora multifunción
10 Unai Cable USB-C
11 Ane Mochila para portátil
12 Jon Monitor 24 pulgadas
13 Nerea Micrófono USB
14 Mikel Tarjeta gráfica
15 Leire Hub USB-C
16 Iker Disco duro externo 2 TB
17 Maialen Silla ergonómica
18 Unai Teclado inalámbrico
19 Mikel Memoria RAM 16 GB
20 Nerea Tablet 10 pulgadas
OK: 20 pedidos recuperados


## Ejercicio 4

In [20]:
# 4.1-4.3 guardar el array de objetos en /pedidos/salida/pedidos_recibidos.json
with client.write(
    '/user/iabd-user15/pedidos/salida/pedidos_recibidos.json',
    encoding='utf-8',
    overwrite=True
) as writer:
    writer.write(json.dumps(list(pedidos_recibidos.values()), ensure_ascii=False, indent=2))

## Ejercicio 5

In [21]:
# 5.1 leer pedidos_recibidos.json desde HDFS (sin Pandas)
with client.read(
    '/user/iabd-user15/pedidos/salida/pedidos_recibidos.json',
    encoding='utf-8'
) as reader:
    pedidos_final = json.load(reader)

# 4.4 comparar los identificadores recibidos y verificar la escritura
ids_escritos = set(pedido["id_pedido"] for pedido in pedidos_final)
ids_recibidos = set(pedidos_recibidos.keys())

print(len(pedidos_final), "pedidos en el archivo")
print("Ids coinciden:", ids_escritos == ids_recibidos)
print("Sin duplicados:", len(pedidos_final) == len(ids_escritos))

20 pedidos en el archivo
Ids coinciden: True
Sin duplicados: True


### Comprobar

In [22]:
# 5.2-5.6 calcular las metricas desde HDFS
importe_total = 0
max_importe = 0
pedido_mayor = None
por_categoria = {}
ids = set()

for pedido in pedidos_final:
    importe = pedido["cantidad"] * pedido["precio_unitario"]
    importe_total += importe

    if importe > max_importe:
        max_importe = importe
        pedido_mayor = pedido

    categoria = pedido["categoria"]
    if categoria in por_categoria:
        por_categoria[categoria] += 1
    else:
        por_categoria[categoria] = 1

    ids.add(pedido["id_pedido"])

ids_originales = set(pedido["id_pedido"] for pedido in pedidos)

print("Numero total de pedidos:", len(pedidos_final))          # 5.2
print("Importe total:", round(importe_total, 2))               # 5.3
print("Pedido con mayor importe:", pedido_mayor["id_pedido"],  # 5.4
      pedido_mayor["producto"], max_importe)
print("Pedidos por categoria:", por_categoria)                 # 5.5
print("Identificadores completos:", ids == ids_originales)     # 5.6

Numero total de pedidos: 20
Importe total: 3364.81
Pedido con mayor importe: 1 Portátil 850
Pedidos por categoria: {'Informática': 3, 'Accesorios': 6, 'Audio': 2, 'Almacenamiento': 2, 'Periféricos': 1, 'Oficina': 1, 'Complementos': 1, 'Componentes': 2, 'Mobiliario': 1, 'Dispositivos móviles': 1}
Identificadores completos: True
